In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

print("Libraries imported successfully!")

In [ ]:
# ============================================================
# 02. DATA LOADING - OLIST E-COMMERCE ANALYTICS
# ============================================================

import pandas as pd
from pathlib import Path

# Define project directory
data_path = Path.cwd().parent  # Run notebook from the notebooks/ folder

# Identify CSV files
csv_files = sorted((data_path / "data" / "raw").glob("*.csv"))

# Load all datasets
datasets = {}

for file in csv_files:
    name = file.stem.replace("olist_", "").replace("_dataset", "")

    datasets[name] = pd.read_csv(file)

    print(f"{name}: {datasets[name].shape[0]:,} rows")

print("-" * 50)
print(f"Total datasets loaded: {len(datasets)}")

In [ ]:
display(datasets["orders"].head())

In [ ]:
# ============================================================
# PHASE 02 - DATA CLEANING & VALIDATION
# 01. CREATE WORKING COPIES
# ============================================================

cleaned_datasets = {
    name: df.copy(deep=True)
    for name, df in datasets.items()
}

print(f"Working copies created: {len(cleaned_datasets)}")

# Step 2: Examining Dataset Sturcture

In [ ]:
for name, df in cleaned_datasets.items():

    print(f"\n{'=' * 60}")
    print(f"TABLE: {name.upper()}")
    print(f"{'=' * 60}")

    print(f"Rows: {df.shape[0]:,}")
    print(f"Columns: {df.shape[1]}")

    display(pd.DataFrame({
        "Column": df.columns,
        "Data Type": df.dtypes.astype(str).values,
        "Non-Null Count": df.notna().sum().values
    }))

In [ ]:
# ============================================================
# 03. INITIAL DATA QUALITY AUDIT
# ============================================================

quality_report = []

for name, df in cleaned_datasets.items():

    total_cells = df.shape[0] * df.shape[1]
    missing_count = df.isna().sum().sum()

    quality_report.append({
        "Dataset": name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Missing Values": missing_count,
        "Missing (%)": round(
            missing_count / total_cells * 100, 2
        ) if total_cells else 0,
        "Duplicate Rows": df.duplicated().sum(),
        "Duplicate (%)": round(
            df.duplicated().mean() * 100, 2
        ) if len(df) else 0
    })

quality_df = pd.DataFrame(quality_report)

display(quality_df)

In [ ]:
# ============================================================
# 04. COLUMN-LEVEL MISSING VALUE ANALYSIS
# ============================================================

missing_report = []

for name, df in cleaned_datasets.items():

    for column in df.columns:

        missing_count = df[column].isna().sum()

        if missing_count > 0:

            missing_report.append({
                "Dataset": name,
                "Column": column,
                "Missing Count": missing_count,
                "Missing (%)": round(
                    missing_count / len(df) * 100, 2
                )
            })

missing_df = pd.DataFrame(missing_report)

if not missing_df.empty:
    display(
        missing_df.sort_values(
            "Missing Count",
            ascending=False
        )
    )
else:
    print("No missing values detected.")

In [ ]:
# ============================================================
# 05. PRIMARY KEY VALIDATION
# ============================================================

primary_keys = {
    "customers": ["customer_id"],
    "orders": ["order_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "order_items": ["order_id", "order_item_id"],
    "order_payments": ["order_id", "payment_sequential"],
    "product_category_name_translation": ["product_category_name"]
}

pk_results = []

for table, keys in primary_keys.items():

    df = cleaned_datasets[table]

    duplicate_keys = df.duplicated(
        subset=keys,
        keep=False
    ).sum()

    null_keys = df[keys].isna().any(axis=1).sum()

    pk_results.append({
        "Dataset": table,
        "Primary Key": ", ".join(keys),
        "Duplicate Key Rows": duplicate_keys,
        "Null Key Rows": null_keys,
        "Status": (
            "PASS"
            if duplicate_keys == 0 and null_keys == 0
            else "REVIEW"
        )
    })

pk_df = pd.DataFrame(pk_results)

display(pk_df)

In [ ]:
# ============================================================
# 06. EXPORT INITIAL QUALITY REPORTS
# ============================================================

report_dir = data_path / "reports"
report_dir.mkdir(parents=True, exist_ok=True)

quality_df.to_csv(
    report_dir / "initial_quality_report.csv",
    index=False
)

missing_df.to_csv(
    report_dir / "missing_values_report.csv",
    index=False
)

pk_df.to_csv(
    report_dir / "primary_key_validation.csv",
    index=False
)

print("Initial data quality reports saved successfully!")

In [ ]:
# ============================================================
# 07. DATE & TIME STANDARDIZATION
# ============================================================

date_columns = {
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],
    "order_items": [
        "shipping_limit_date"
    ],
    "order_reviews": [
        "review_creation_date",
        "review_answer_timestamp"
    ]
}

for table, columns in date_columns.items():

    df = cleaned_datasets[table]

    for column in columns:

        original = df[column].copy()

        df[column] = pd.to_datetime(
            original,
            errors="coerce"
        )

        new_invalid = (
            original.notna() & df[column].isna()
        ).sum()

        print(
            f"{table}.{column}: "
            f"{df[column].dtype} | "
            f"New invalid dates: {new_invalid}"
        )

In [ ]:
# ============================================================
# 08. MISSING DELIVERY DATE INVESTIGATION
# ============================================================

orders_clean = cleaned_datasets["orders"]

delivery_analysis = (
    orders_clean
    .groupby("order_status", dropna=False)
    .agg(
        total_orders=("order_id", "count"),
        missing_delivery_dates=(
            "order_delivered_customer_date",
            lambda x: x.isna().sum()
        )
    )
    .reset_index()
)

delivery_analysis["missing_pct"] = (
    delivery_analysis["missing_delivery_dates"]
    / delivery_analysis["total_orders"] * 100
).round(2)

display(delivery_analysis)

In [ ]:
# ============================================================
# 09. PRODUCT ATTRIBUTE VALIDATION
# ============================================================

products_clean = cleaned_datasets["products"]

attribute_columns = [
    "product_category_name",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty"
]

missing_patterns = (
    products_clean[attribute_columns]
    .isna()
    .value_counts()
    .reset_index(name="product_count")
)

display(missing_patterns)

print(
    "Products missing all four attributes:",
    products_clean[attribute_columns].isna().all(axis=1).sum()
)

In [ ]:
# ============================================================
# 10. GEOLOCATION DUPLICATE INVESTIGATION
# ============================================================

geo = cleaned_datasets["geolocation"]

print("Total rows:", len(geo))
print("Exact duplicate rows:", geo.duplicated().sum())

print(
    "Unique ZIP prefixes:",
    geo["geolocation_zip_code_prefix"].nunique()
)

zip_frequency = (
    geo.groupby("geolocation_zip_code_prefix")
    .size()
    .sort_values(ascending=False)
    .head(10)
)

display(zip_frequency)

In [ ]:
# Remove exact duplicate observations for lookup preparation
geo_unique = geo.drop_duplicates().copy()

# Calculate representative coordinates per ZIP prefix
geo_lookup = (
    geo_unique
    .groupby("geolocation_zip_code_prefix", as_index=False)
    .agg(
        latitude=("geolocation_lat", "median"),
        longitude=("geolocation_lng", "median")
    )
)

print("Unique location records:", len(geo_unique))
print("ZIP lookup rows:", len(geo_lookup))

display(geo_lookup.head())

In [ ]:
# ============================================================
# 11. FOREIGN KEY INTEGRITY VALIDATION
# ============================================================

relationships = [
    ("orders", "customer_id", "customers", "customer_id"),
    ("order_items", "order_id", "orders", "order_id"),
    ("order_items", "product_id", "products", "product_id"),
    ("order_items", "seller_id", "sellers", "seller_id"),
    ("order_payments", "order_id", "orders", "order_id"),
    ("order_reviews", "order_id", "orders", "order_id")
]

fk_results = []

for child, child_key, parent, parent_key in relationships:

    child_df = cleaned_datasets[child]
    parent_df = cleaned_datasets[parent]

    parent_values = parent_df[parent_key].dropna().unique()

    orphan_mask = (
        child_df[child_key].notna()
        & ~child_df[child_key].isin(parent_values)
    )

    orphan_count = orphan_mask.sum()

    fk_results.append({
        "Child Table": child,
        "Foreign Key": child_key,
        "Parent Table": parent,
        "Orphan Records": orphan_count,
        "Status": "PASS" if orphan_count == 0 else "REVIEW"
    })

fk_df = pd.DataFrame(fk_results)

display(fk_df)

In [ ]:
# ============================================================
# 12. DELIVERY TIMESTAMP ANOMALIES
# ============================================================

orders_clean = cleaned_datasets["orders"]

delivery_anomalies = orders_clean.loc[
    (orders_clean["order_status"] == "delivered")
    & (orders_clean["order_delivered_customer_date"].isna())
]

print("Delivered orders missing delivery timestamp:",
      len(delivery_anomalies))

display(delivery_anomalies)

In [ ]:
canceled_delivered = orders_clean.loc[
    (orders_clean["order_status"] == "canceled")
    & (orders_clean["order_delivered_customer_date"].notna())
]

print("Canceled orders with delivery timestamps:",
      len(canceled_delivered))

display(canceled_delivered)

In [ ]:
# ============================================================
# 13. NUMERIC BUSINESS RULE VALIDATION
# ============================================================

numeric_checks = {
    "products": [
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ],
    "order_items": [
        "price",
        "freight_value"
    ],
    "order_payments": [
        "payment_value"
    ]
}

numeric_results = []

for table, columns in numeric_checks.items():

    df = cleaned_datasets[table]

    for col in columns:

        values = pd.to_numeric(df[col], errors="coerce")

        numeric_results.append({
            "Dataset": table,
            "Column": col,
            "Negative Values": int((values < 0).sum()),
            "Zero Values": int((values == 0).sum()),
            "Missing Values": int(values.isna().sum())
        })

numeric_df = pd.DataFrame(numeric_results)

display(numeric_df)

In [ ]:
# ============================================================
# 14. GEOLOCATION COORDINATE VALIDATION
# ============================================================

geo = cleaned_datasets["geolocation"]

# Broad bounding box for Brazil
invalid_coordinates = geo.loc[
    ~geo["geolocation_lat"].between(-34, 6)
    | ~geo["geolocation_lng"].between(-74, -28)
]

print("Coordinates outside broad Brazil bounds:",
      len(invalid_coordinates))

display(invalid_coordinates.head(10))

In [ ]:
# ============================================================
# 15. ORDER DATA QUALITY FLAGS
# ============================================================

orders_clean = cleaned_datasets["orders"]

orders_clean["dq_missing_delivery_date"] = (
    orders_clean["order_status"].eq("delivered")
    & orders_clean["order_delivered_customer_date"].isna()
)

orders_clean["dq_canceled_but_delivered"] = (
    orders_clean["order_status"].eq("canceled")
    & orders_clean["order_delivered_customer_date"].notna()
)

print(
    "Delivered orders missing dates:",
    orders_clean["dq_missing_delivery_date"].sum()
)

print(
    "Canceled orders with delivery dates:",
    orders_clean["dq_canceled_but_delivered"].sum()
)

In [ ]:
# ============================================================
# 16. ZERO PAYMENT INVESTIGATION
# ============================================================

payments_clean = cleaned_datasets["order_payments"]

zero_payments = payments_clean.loc[
    payments_clean["payment_value"] == 0
]

print("Zero-value payment records:", len(zero_payments))

display(
    zero_payments.groupby(
        "payment_type", dropna=False
    ).size().reset_index(name="record_count")
)

display(zero_payments.head(10))

In [ ]:
# ============================================================
# 17. PRODUCT DATA QUALITY FLAGS
# ============================================================

products_clean = cleaned_datasets["products"]

attribute_columns = [
    "product_category_name",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty"
]

products_clean["dq_missing_attributes"] = (
    products_clean[attribute_columns].isna().all(axis=1)
)

products_clean["dq_invalid_weight"] = (
    products_clean["product_weight_g"].notna()
    & (products_clean["product_weight_g"] <= 0)
)

print(
    "Products missing core attributes:",
    products_clean["dq_missing_attributes"].sum()
)

print(
    "Products with zero/negative weight:",
    products_clean["dq_invalid_weight"].sum()
)

In [ ]:
# ============================================================
# 18. CLEAN GEOGRAPHICAL LOOKUP
# ============================================================

geo = cleaned_datasets["geolocation"].copy()

# Approximate Brazil coordinate screening
valid_geo = geo.loc[
    geo["geolocation_lat"].between(-34, 6)
    & geo["geolocation_lng"].between(-74, -28)
].copy()

# Remove exact repeated observations
valid_geo = valid_geo.drop_duplicates()

# One representative coordinate per ZIP prefix
geo_lookup = (
    valid_geo
    .groupby("geolocation_zip_code_prefix", as_index=False)
    .agg(
        latitude=("geolocation_lat", "median"),
        longitude=("geolocation_lng", "median")
    )
)

print("Original geolocation rows:", len(geo))
print("Coordinate-screened unique rows:", len(valid_geo))
print("Geolocation lookup rows:", len(geo_lookup))

display(geo_lookup.head())

In [ ]:
# ============================================================
# 19. ORDER FINANCIAL RECONCILIATION
# ============================================================

items_summary = (
    cleaned_datasets["order_items"]
    .groupby("order_id", as_index=False)
    .agg(
        items_value=("price", "sum"),
        freight_value=("freight_value", "sum")
    )
)

items_summary["expected_total"] = (
    items_summary["items_value"]
    + items_summary["freight_value"]
)

payments_summary = (
    cleaned_datasets["order_payments"]
    .groupby("order_id", as_index=False)
    .agg(
        payment_total=("payment_value", "sum")
    )
)

financial_check = items_summary.merge(
    payments_summary,
    on="order_id",
    how="outer",
    indicator=True
)

financial_check["difference"] = (
    financial_check["payment_total"]
    - financial_check["expected_total"]
)

financial_check["is_reconciled"] = (
    financial_check["_merge"].eq("both")
    & financial_check["difference"].abs().le(0.01)
)

print("Orders examined:", len(financial_check))
print("Reconciled orders:", financial_check["is_reconciled"].sum())
print("Orders requiring review:", (~financial_check["is_reconciled"]).sum())

display(
    financial_check.loc[
        ~financial_check["is_reconciled"]
    ].head(10)
)

In [ ]:
# ============================================================
# 20. FINANCIAL DISCREPANCY CLASSIFICATION
# ============================================================

financial_check["reconciliation_status"] = np.select(
    [
        financial_check["_merge"].eq("left_only"),
        financial_check["_merge"].eq("right_only"),
        financial_check["is_reconciled"]
    ],
    [
        "Missing Payment Record",
        "Missing Order Items",
        "Reconciled"
    ],
    default="Amount Mismatch"
)

reconciliation_summary = (
    financial_check
    .groupby("reconciliation_status", as_index=False)
    .agg(
        order_count=("order_id", "count")
    )
)

reconciliation_summary["percentage"] = (
    reconciliation_summary["order_count"]
    / len(financial_check) * 100
).round(2)

display(reconciliation_summary)

In [ ]:
# ============================================================
# 21. RECONCILIATION BY ORDER STATUS
# ============================================================

financial_analysis = financial_check.merge(
    cleaned_datasets["orders"][
        ["order_id", "order_status"]
    ],
    on="order_id",
    how="left",
    validate="one_to_one"
)

status_summary = (
    financial_analysis
    .groupby(
        ["order_status", "reconciliation_status"],
        dropna=False
    )
    .size()
    .reset_index(name="order_count")
)

display(
    status_summary.sort_values(
        "order_count",
        ascending=False
    )
)

In [ ]:
# ============================================================
# 22. FINANCIAL AMOUNT MISMATCH INVESTIGATION
# ============================================================

amount_mismatches = financial_analysis.loc[
    financial_analysis["reconciliation_status"]
    == "Amount Mismatch"
].copy()

amount_mismatches["absolute_difference"] = (
    amount_mismatches["difference"].abs()
)

display(
    amount_mismatches[
        [
            "order_id",
            "order_status",
            "expected_total",
            "payment_total",
            "difference",
            "absolute_difference"
        ]
    ]
    .sort_values(
        "absolute_difference",
        ascending=False
    )
    .head(20)
)

print("Amount mismatches:", len(amount_mismatches))

In [ ]:
# ============================================================
# 23. GEOGRAPHICAL LOOKUP COVERAGE
# ============================================================

original_zips = set(
    cleaned_datasets["geolocation"][
        "geolocation_zip_code_prefix"
    ].dropna()
)

valid_zips = set(
    geo_lookup["geolocation_zip_code_prefix"]
)

missing_zips = sorted(original_zips - valid_zips)

print("Original ZIP prefixes:", len(original_zips))
print("Lookup ZIP prefixes:", len(valid_zips))
print("ZIP prefixes excluded:", len(missing_zips))
print("Excluded ZIP prefixes:", missing_zips)

In [ ]:
# ============================================================
# 24. FINANCIAL QUALITY CLASSIFICATION
# ============================================================

orders_clean = cleaned_datasets["orders"]

# Add reconciliation status to each order
orders_clean = orders_clean.merge(
    financial_check[
        ["order_id", "reconciliation_status", "difference"]
    ],
    on="order_id",
    how="left",
    validate="one_to_one"
)

orders_clean["dq_financial_exception"] = (
    orders_clean["reconciliation_status"] != "Reconciled"
)

cleaned_datasets["orders"] = orders_clean

display(
    orders_clean["reconciliation_status"]
    .value_counts(dropna=False)
)

In [ ]:
# ============================================================
# 25. FINAL FINANCIAL SUMMARY
# ============================================================

financial_summary = pd.DataFrame({
    "Metric": [
        "Total item value",
        "Total freight value",
        "Total payment value",
        "Financial exceptions"
    ],
    "Value": [
        cleaned_datasets["order_items"]["price"].sum(),
        cleaned_datasets["order_items"]["freight_value"].sum(),
        cleaned_datasets["order_payments"]["payment_value"].sum(),
        orders_clean["dq_financial_exception"].sum()
    ]
})

display(financial_summary)

In [ ]:
# ============================================================
# 26. REVIEW UNIQUENESS VALIDATION
# ============================================================

reviews = cleaned_datasets["order_reviews"]

print("Total review rows:", len(reviews))
print("Unique review IDs:", reviews["review_id"].nunique())
print("Duplicate review IDs:",
      reviews["review_id"].duplicated().sum())

print("Duplicate review/order combinations:",
      reviews.duplicated(
          subset=["review_id", "order_id"]
      ).sum())

In [ ]:
# ============================================================
# 27. FINAL PRE-EXPORT VALIDATION
# ============================================================

# Verify all nine datasets are present
assert len(cleaned_datasets) == 9, (
    "Expected exactly nine datasets."
)

# Check for missing review composite-key values
reviews = cleaned_datasets["order_reviews"]

review_null_keys = reviews[
    ["review_id", "order_id"]
].isna().any(axis=1).sum()

review_duplicate_keys = reviews.duplicated(
    subset=["review_id", "order_id"]
).sum()

print("Missing review key rows:", review_null_keys)
print("Duplicate review composite keys:", review_duplicate_keys)

assert review_null_keys == 0
assert review_duplicate_keys == 0

# Ensure all original rows are preserved
for name, original_df in datasets.items():

    cleaned_df = cleaned_datasets[name]

    assert len(original_df) == len(cleaned_df), (
        f"Row count changed unexpectedly: {name}"
    )

print("All nine source-table row counts preserved.")

In [ ]:
# ============================================================
# 28. EXPORT CLEANED DATASETS
# ============================================================

from pathlib import Path

# Use the project directory established earlier
output_dir = data_path / "data" / "cleaned"

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

export_summary = []

for name, df in cleaned_datasets.items():

    file_path = output_dir / f"{name}_cleaned.csv"

    df.to_csv(
        file_path,
        index=False,
        encoding="utf-8-sig",
        date_format="%Y-%m-%d %H:%M:%S"
    )

    export_summary.append({
        "Dataset": name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "File": file_path.name
    })

export_df = pd.DataFrame(export_summary)

display(export_df)

print("All nine datasets exported successfully!")

In [ ]:
# ============================================================
# 29. EXPORT GEOGRAPHICAL LOOKUP
# ============================================================

lookup_path = output_dir / "geolocation_lookup.csv"

assert geo_lookup[
    "geolocation_zip_code_prefix"
].is_unique

geo_lookup.to_csv(
    lookup_path,
    index=False,
    encoding="utf-8-sig"
)

print("Geolocation lookup exported:", len(geo_lookup))

In [ ]:
# ============================================================
# 30. EXPORT FINAL VALIDATION REPORTS
# ============================================================

report_dir = data_path / "reports"

report_dir.mkdir(
    parents=True,
    exist_ok=True
)

reconciliation_summary.to_csv(
    report_dir / "financial_reconciliation_summary.csv",
    index=False
)

financial_analysis.to_csv(
    report_dir / "financial_reconciliation_details.csv",
    index=False
)

fk_df.to_csv(
    report_dir / "foreign_key_validation.csv",
    index=False
)

numeric_df.to_csv(
    report_dir / "numeric_validation.csv",
    index=False
)

export_df.to_csv(
    report_dir / "cleaned_dataset_manifest.csv",
    index=False
)

print("Validation reports exported successfully!")

In [ ]:
print("Cleaned CSV folder:", output_dir.resolve())
print("Validation reports folder:", report_dir.resolve())